In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

print("🧪 Starting Silver (Map Stage) Unit Tests...")

test_catalog = "hgi"
test_schema = "silver_test"

# ===================================================================================
# 1. Setup Mock Unified Silver Tables (Initial State)
# ===================================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {test_catalog}.{test_schema}")

# Clean up previous test runs
tables_to_drop = ["unified_accounts", "unified_contacts", "unified_tasks", "unified_events", "crm_events", "events", "contacts_to_accounts", "email_events_mapped", "domain_events_mapped", "mk_account_events_mapped", "accounts_attributes", "contacts_attributes"]
for t in tables_to_drop:
    spark.sql(f"DROP TABLE IF EXISTS {test_catalog}.{test_schema}.{t}")

# Create Base Unified Tables
spark.sql(f"CREATE TABLE {test_catalog}.{test_schema}.unified_accounts (tenant BIGINT, id STRING, domain STRING, source_system STRING) USING DELTA")
spark.sql(f"CREATE TABLE {test_catalog}.{test_schema}.unified_contacts (tenant BIGINT, id STRING, email STRING, domain STRING, a_accountid STRING, source_system STRING) USING DELTA")
spark.sql(f"CREATE TABLE {test_catalog}.{test_schema}.unified_tasks (tenant BIGINT, id STRING, contact_source_system_object STRING, contact_source_key_value STRING, a_subject STRING, event_timestamp TIMESTAMP, source_system STRING) USING DELTA")
spark.sql(f"CREATE TABLE {test_catalog}.{test_schema}.unified_events (tenant BIGINT, id STRING, event STRING, event_timestamp TIMESTAMP, contact_source_key_value STRING, domains STRING, source_system STRING) USING DELTA")

# ===================================================================================
# 2. Insert Mock Data (Edge Cases from PDF)
# ===================================================================================
# Accounts
spark.sql(f"INSERT INTO {test_catalog}.{test_schema}.unified_accounts VALUES (1, 'salesforce_Account_Id_ACC1', 'company.com', 'salesforce')")

# Contacts (Testing 3-Phase Logic) [cite: 588-605]
spark.sql(f"""
    INSERT INTO {test_catalog}.{test_schema}.unified_contacts VALUES 
    (1, 'salesforce_Contact_Id_C1', 'alice@explicit.com', 'explicit.com', 'ACC1', 'salesforce'),      -- Phase 1: CRM Defined (Has AccountId)
    (1, 'salesforce_Contact_Id_C2', 'bob@company.com', 'company.com', NULL, 'salesforce'),            -- Phase 2: Domain Match (Matches ACC1)
    (1, 'salesforce_Contact_Id_C3', 'charlie@startup.io', 'startup.io', NULL, 'salesforce'),          -- Phase 2b: Domain Created (No matching account)
    (1, 'salesforce_Contact_Id_C4', 'dave@gmail.com', 'gmail.com', NULL, 'salesforce')                -- Phase 3: Free Email Handling
""")

# Tasks (Testing CRM Events Mapping & Subject Regex) 
spark.sql(f"""
    INSERT INTO {test_catalog}.{test_schema}.unified_tasks VALUES 
    (1, 'salesforce_Task_Id_T1', 'Contact', 'C1', 'Call', CAST('2026-03-21T10:00:00Z' AS TIMESTAMP), 'salesforce'),
    (1, 'salesforce_Task_Id_T2', 'Contact', 'C1', 'Email: Re: Pricing', CAST('2026-03-21T11:00:00Z' AS TIMESTAMP), 'salesforce'),
    (1, 'salesforce_Task_Id_T3', 'Contact', 'C2', 'Email: Intro', CAST('2026-03-21T12:00:00Z' AS TIMESTAMP), 'salesforce'),
    (1, 'salesforce_Task_Id_T4', 'Contact', 'C2', 'Random Meeting', CAST('2026-03-21T13:00:00Z' AS TIMESTAMP), 'salesforce')
""")

# BQ Events (Testing Anonymous vs Identified) [cite: 612-614]
spark.sql(f"""
    INSERT INTO {test_catalog}.{test_schema}.unified_events VALUES 
    (1, 'bigquery_Event_E1', 'page_view', CAST('2026-03-21T14:00:00Z' AS TIMESTAMP), 'alice@explicit.com', 'explicit.com', 'bigquery'), -- Identified
    (1, 'bigquery_Event_E2', 'page_view', CAST('2026-03-21T15:00:00Z' AS TIMESTAMP), NULL, 'company.com', 'bigquery')                   -- Anonymous
""")

print("✅ Mock Data Inserted.")

# ===================================================================================
# 3. Execute Map Layer Logic
# ===================================================================================
print("⚙️ Executing Map Stage SQL...")

# Views
spark.sql(f"CREATE OR REPLACE VIEW {test_catalog}.{test_schema}.contacts_all AS SELECT id, email, domain, source_system, tenant FROM {test_catalog}.{test_schema}.unified_contacts")
spark.sql(f"CREATE OR REPLACE VIEW {test_catalog}.{test_schema}.accounts_all AS SELECT id, domain, source_system, tenant FROM {test_catalog}.{test_schema}.unified_accounts")

# CRM Events
spark.sql(f"""
    CREATE OR REPLACE TABLE {test_catalog}.{test_schema}.crm_events USING DELTA AS
    SELECT id, tenant, event_timestamp, concat('salesforce_', contact_source_system_object, '_Id_', contact_source_key_value) AS contact_id,
        CASE WHEN a_subject = 'Call' THEN 'call' WHEN a_subject LIKE 'Email: Re:%' THEN 'email_reply' WHEN a_subject LIKE 'Email:%' THEN 'email_sent' ELSE 'other' END AS meta_event
    FROM {test_catalog}.{test_schema}.unified_tasks WHERE source_system = 'salesforce' AND contact_source_key_value IS NOT NULL
""")

# Events Mapping
spark.sql(f"""
    CREATE OR REPLACE TABLE {test_catalog}.{test_schema}.events USING DELTA AS
    SELECT e.id, e.tenant, e.event_timestamp, COALESCE(e.event, 'other') AS meta_event, COALESCE(c.id, concat(e.source_system, '_Contact_Id_', e.contact_source_key_value)) AS contactid, e.domains AS domain
    FROM {test_catalog}.{test_schema}.unified_events e
    LEFT JOIN {test_catalog}.{test_schema}.contacts_all c ON c.email = e.contact_source_key_value OR c.id = concat(e.source_system, '_Contact_Id_', e.contact_source_key_value)
    UNION ALL
    SELECT ce.id, ce.tenant, ce.event_timestamp, ce.meta_event, ce.contact_id AS contactid, c.domain AS domain
    FROM {test_catalog}.{test_schema}.crm_events ce
    LEFT JOIN {test_catalog}.{test_schema}.contacts_all c ON ce.contact_id = c.id
""")

# Contacts to Accounts 
spark.sql(f"""
    CREATE OR REPLACE TABLE {test_catalog}.{test_schema}.contacts_to_accounts USING DELTA AS
    SELECT c.id AS contact_id, concat('salesforce_Account_Id_', c.a_accountid) AS account_id, 'crm_defined' AS match_type
    FROM {test_catalog}.{test_schema}.unified_contacts c WHERE c.a_accountid IS NOT NULL
    UNION ALL
    SELECT c.id AS contact_id, CASE WHEN c.domain IN ('gmail.com', 'yahoo.com', 'hotmail.com', 'aol.com') THEN concat('madkudu_map_email_', c.email) WHEN a.id IS NOT NULL THEN a.id ELSE concat('madkudu_map_domain_', c.domain) END AS account_id,
        CASE WHEN c.domain IN ('gmail.com', 'yahoo.com', 'hotmail.com', 'aol.com') THEN 'free_email_match' WHEN a.id IS NOT NULL THEN 'domain_match' ELSE 'domain_created' END AS match_type
    FROM {test_catalog}.{test_schema}.unified_contacts c
    LEFT JOIN (SELECT domain, FIRST(id) as id FROM {test_catalog}.{test_schema}.unified_accounts WHERE domain IS NOT NULL GROUP BY domain) a ON c.domain = a.domain
    WHERE c.a_accountid IS NULL AND c.domain IS NOT NULL
""")

# Aggregations
spark.sql(f"CREATE OR REPLACE TABLE {test_catalog}.{test_schema}.email_events_mapped USING DELTA AS SELECT c.email, e.meta_event, CAST(e.event_timestamp AS DATE) AS event_day, COUNT(*) AS occurrences FROM {test_catalog}.{test_schema}.events e JOIN {test_catalog}.{test_schema}.contacts_all c ON e.contactid = c.id WHERE c.email IS NOT NULL GROUP BY c.email, e.meta_event, CAST(e.event_timestamp AS DATE)")
spark.sql(f"CREATE OR REPLACE TABLE {test_catalog}.{test_schema}.domain_events_mapped USING DELTA AS SELECT c.domain, e.meta_event, CAST(e.event_timestamp AS DATE) AS event_day, COUNT(*) AS occurrences FROM {test_catalog}.{test_schema}.events e JOIN {test_catalog}.{test_schema}.contacts_all c ON e.contactid = c.id WHERE c.domain IS NOT NULL GROUP BY c.domain, e.meta_event, CAST(e.event_timestamp AS DATE)")
spark.sql(f"CREATE OR REPLACE TABLE {test_catalog}.{test_schema}.mk_account_events_mapped USING DELTA AS SELECT c2a.account_id AS mk_accountid_domain, e.meta_event, CAST(e.event_timestamp AS DATE) AS event_day, COUNT(*) AS occurrences, false AS anonymous_activity FROM {test_catalog}.{test_schema}.events e JOIN {test_catalog}.{test_schema}.contacts_to_accounts c2a ON e.contactid = c2a.contact_id WHERE e.contactid IS NOT NULL GROUP BY 1, 2, 3, 5 UNION ALL SELECT a.id AS mk_accountid_domain, e.meta_event, CAST(e.event_timestamp AS DATE) AS event_day, COUNT(*) AS occurrences, true AS anonymous_activity FROM {test_catalog}.{test_schema}.events e JOIN {test_catalog}.{test_schema}.accounts_all a ON e.domain = a.domain WHERE e.contactid IS NULL AND e.domain IS NOT NULL GROUP BY 1, 2, 3, 5")

# Attributes [cite: 809]
spark.sql(f"CREATE OR REPLACE TABLE {test_catalog}.{test_schema}.accounts_attributes USING DELTA AS SELECT id AS account_id, false AS is_paying, false AS is_excluded, 0.0 AS mrr FROM {test_catalog}.{test_schema}.accounts_all")
spark.sql(f"CREATE OR REPLACE TABLE {test_catalog}.{test_schema}.contacts_attributes USING DELTA AS SELECT id AS contact_id, false AS is_paying, false AS is_excluded, 0.0 AS mrr FROM {test_catalog}.{test_schema}.contacts_all")


# ===================================================================================
# 4. Validation & Assertions
# ===================================================================================
print("\n📊 Validating Results against Client PDF Rules...\n")
errors = []

# Validate TC1: CRM Events Mapping 
crm_events = spark.table(f"{test_catalog}.{test_schema}.crm_events").collect()
event_map = {row["id"]: row["meta_event"] for row in crm_events}
if event_map.get("salesforce_Task_Id_T1") != "call": errors.append("❌ TC1 Failed: 'Call' not mapped to 'call'")
if event_map.get("salesforce_Task_Id_T2") != "email_reply": errors.append("❌ TC1 Failed: 'Email: Re:' not mapped to 'email_reply'")
if event_map.get("salesforce_Task_Id_T3") != "email_sent": errors.append("❌ TC1 Failed: 'Email:' not mapped to 'email_sent'")
if event_map.get("salesforce_Task_Id_T4") != "other": errors.append("❌ TC1 Failed: Random subject not mapped to 'other'")

# Validate TC2: Contacts-to-Accounts 3-Phase Logic [cite: 588-605]
c2a = spark.table(f"{test_catalog}.{test_schema}.contacts_to_accounts").collect()
c2a_map = {row["contact_id"]: (row["account_id"], row["match_type"]) for row in c2a}

if c2a_map.get("salesforce_Contact_Id_C1")[1] != "crm_defined": 
    errors.append("❌ TC2 Failed: Phase 1 (CRM Defined) failed.")
if c2a_map.get("salesforce_Contact_Id_C2")[0] != "salesforce_Account_Id_ACC1": 
    errors.append("❌ TC2 Failed: Phase 2 (Domain Match) failed to link to existing account.")
if c2a_map.get("salesforce_Contact_Id_C3")[1] != "domain_created": 
    errors.append("❌ TC2 Failed: Phase 2b (Domain Created) failed for unknown domain.")
if c2a_map.get("salesforce_Contact_Id_C4")[1] != "free_email_match": 
    errors.append("❌ TC2 Failed: Phase 3 (Free Email) failed to flag gmail.com.")

# Validate TC3: Event Aggregations (Anonymous vs Identified) [cite: 612-614]
mk_events = spark.table(f"{test_catalog}.{test_schema}.mk_account_events_mapped").collect()
for row in mk_events:
    if row["mk_accountid_domain"] == "salesforce_Account_Id_ACC1" and row["meta_event"] == "page_view":
        if not row["anonymous_activity"]:
            identified_found = True # E1
        if row["anonymous_activity"]:
            anonymous_found = True  # E2

if not identified_found or not anonymous_found:
    errors.append("❌ TC3 Failed: Account rollup did not correctly separate identified and anonymous activity.")

# Validate TC4: Output Expectations - No Orphans 
orphan_count = spark.sql(f"""
    SELECT COUNT(*) as cnt FROM {test_catalog}.{test_schema}.contacts_to_accounts c2a 
    LEFT JOIN {test_catalog}.{test_schema}.contacts_all c ON c2a.contact_id = c.id 
    WHERE c.id IS NULL
""").first()["cnt"]
if orphan_count > 0: errors.append(f"❌ TC4 Failed: Found {orphan_count} orphans in contacts_to_accounts.")

# Validate TC5: Output Expectations - Attributes Coverage [cite: 809]
missing_attr = spark.sql(f"""
    SELECT COUNT(*) as cnt FROM {test_catalog}.{test_schema}.accounts_all a 
    LEFT JOIN {test_catalog}.{test_schema}.accounts_attributes aa ON a.id = aa.account_id 
    WHERE aa.account_id IS NULL
""").first()["cnt"]
if missing_attr > 0: errors.append(f"❌ TC5 Failed: Found {missing_attr} accounts missing from accounts_attributes.")

if errors:
    for e in errors: print(e)
    raise Exception("Test Suite Failed!")
else:
    print("✅ ALL TEST CASES PASSED!")
    print("✅ 1. CRM Events correctly mapped Task Subjects to 'call', 'email_sent', etc.")
    print("✅ 2. Contacts-to-Accounts logic perfectly executed the 3-phase domain/CRM matching.")
    print("✅ 3. Event Aggregations successfully rolled up signals into Identified and Anonymous buckets.")
    print("✅ 4. Validation Rule: 'contacts_to_accounts has no orphans' passed.")
    print("✅ 5. Validation Rule: 'accounts_attributes covers all accounts' passed.\n")

display(spark.table(f"{test_catalog}.{test_schema}.contacts_to_accounts").orderBy("contact_id"))